In [4]:
# Manipulação de Dados
import pandas as pd
import numpy as np

# Estatística Descritiva
from scipy import stats

# Visualização
import matplotlib.pyplot as plt
import seaborn as sns

# Regressão Linear
from sklearn.linear_model import LinearRegression

# Regressão de Poisson 
from sklearn.linear_model import PoissonRegressor

# Random Forest
from sklearn.ensemble import RandomForestClassifier

# XGBoost
from xgboost import XGBClassifier

# Distancia Geográfica
from geopy.distance import geodesic

In [22]:
df = pd.read_excel(
    "media_fifa.xlsx",
    sheet_name="Planilha1"
)

df_modelo = df.set_index("Paises").T

df_modelo.reset_index(inplace=True)

df_modelo.rename(
    columns={"index": "Selecao"},
    inplace=True
)

df_modelo.head()

Paises,Selecao,Continente,Assistência esperada (xA),Ataque,Cartões amarelos,Cartões vermelhos,Chances perigosas criadas,Chutes bloqueados,Chutes dentro da área,Chutes fora da área,...,Passes no último terço,Passes para trás,Perigo afastado,Posse de Bola,Posse de bola perdida,Posse recuperada no terço final,Tiros de meta,Total de chutes,Trave,xG sofridos (xGC)
0,África do Sul,CAF,0.958571,84,1.9,0.1,2.666667,3.555556,7.555556,6.888889,...,99,77.666667,15.555556,0.636,107.777778,3.111111,6.3,14.444444,0.333333,0.881429
1,Alemanha,UEFA,1.975,99.4,1.5,0,4.3,4.7,12.9,4.8,...,167.3,102.1,16,0.673,115.2,4.9,4.1,17.7,0.7,0.921
2,Argentina,CONMEBOL,1.8475,92.555556,1.111111,0.222222,3.444444,3.222222,7.777778,4.777778,...,152.333333,118.333333,10.333333,0.674444,96.444444,4.333333,4.777778,12.555556,0.333333,0.703333
3,Argélia,CAF,0.805,78.5,1.9,0,2.5,2.1,7.833333,2.333333,...,68.166667,68.833333,19.5,0.608,109.333333,3.5,5.9,11.9,0.222222,1.053333
4,Arábia Saudita,AFC,0,81.857143,1,0.142857,1,2.857143,8,7,...,120,71,38,0.614286,189,3,5.571429,10.142857,0.571429,0


In [23]:
df_modelo.columns.tolist()

['Selecao',
 'Continente',
 'Assistência esperada (xA)',
 'Ataque',
 'Cartões amarelos',
 'Cartões vermelhos',
 'Chances perigosas criadas',
 'Chutes bloqueados',
 'Chutes dentro da área',
 'Chutes fora da área',
 'Chutes no gol',
 'Chutes para fora',
 'Defesas do goleiro',
 'Driblado',
 'Duelos ganhos',
 'Escanteios',
 'Faltas',
 'Faltas sofridas',
 'Gol sofrido devido a erro individual',
 'Gols esperados (xG)',
 'Gols esperados em chutes no alvo',
 'Grandes chances perdidas',
 'Impedimentos',
 'Interceptações',
 'Laterais cobrados',
 'Passes completos',
 'Passes decisivos',
 'Passes no campo adversário',
 'Passes no próprio campo',
 'Passes no último terço',
 'Passes para trás',
 'Perigo afastado',
 'Posse de Bola',
 'Posse de bola perdida',
 'Posse recuperada no terço final',
 'Tiros de meta',
 'Total de chutes',
 'Trave',
 'xG sofridos (xGC)']

In [30]:
colunas_numericas = df2.columns.drop(['Selecao','Continente'])

df2[colunas_numericas] = df2[colunas_numericas].apply(
    pd.to_numeric,
    errors='coerce'
)

In [31]:
print(df2[['Selecao','Ataque','Gols esperados (xG)']].head())

Paises         Selecao     Ataque  Gols esperados (xG)
0        África do Sul  84.000000             1.112857
1             Alemanha  99.400000             2.403000
2            Argentina  92.555556             1.503333
3              Argélia  78.500000             1.231667
4       Arábia Saudita  81.857143             0.000000


In [32]:
colunas_ofensivas = [
    'Ataque',
    'Assistência esperada (xA)',
    'Gols esperados (xG)',
    'Chances perigosas criadas',
    'Chutes dentro da área',
    'Chutes no gol',
    'Passes decisivos',
    'Total de chutes'
]

In [33]:
from scipy.stats import zscore

df_of = df2[colunas_ofensivas]

df_of_z = df_of.apply(zscore)

In [34]:
pesos = {
    'Ataque': 0.20,
    'Assistência esperada (xA)': 0.15,
    'Gols esperados (xG)': 0.20,
    'Chances perigosas criadas': 0.15,
    'Chutes dentro da área': 0.10,
    'Chutes no gol': 0.10,
    'Passes decisivos': 0.05,
    'Total de chutes': 0.05
}

In [35]:
df2['Indice_Ofensivo'] = 0

for coluna, peso in pesos.items():
    df2['Indice_Ofensivo'] += (
        df_of_z[coluna] * peso
    )

In [36]:
ranking_ofensivo = (
    df2[
        ['Selecao','Indice_Ofensivo']
    ]
    .sort_values(
        'Indice_Ofensivo',
        ascending=False
    )
)

print(ranking_ofensivo.head(15))

Paises               Selecao  Indice_Ofensivo
21                   Espanha         2.241271
16                   Croácia         1.651848
8                    Bélgica         1.504674
38                  Portugal         1.378308
1                   Alemanha         1.307921
27                Inglaterra         1.135109
23                    França         1.121655
26                   Holanda         0.973100
6                    Áustria         0.858185
34                   Noruega         0.783583
41                   Senegal         0.658027
9       Bósnia e Herzegovina         0.556280
32                  Marrocos         0.552230
2                  Argentina         0.532613
15           Costa do Marfim         0.403150


In [ ]:
colunas_defensivas = [
    'xG sofridos (xGC)',
    'Defesas do goleiro',
    'Interceptações',
    'Perigo afastado',
    'Posse recuperada no terço final',
    'Duelos ganhos'
]

In [38]:
df2.loc[
    df2['Selecao'].isin(
        ['Argentina','Espanha','França','Alemanha']
    ),
    colunas_ofensivas
]

Paises,Ataque,Assistência esperada (xA),Gols esperados (xG),Chances perigosas criadas,Chutes dentro da área,Chutes no gol,Passes decisivos,Total de chutes
1,99.400000,1.975000,2.403000,4.300000,12.900000,6.300000,14.100000,17.700000
2,92.555556,1.847500,1.503333,3.444444,7.777778,5.555556,10.111111,12.555556
21,128.800000,2.059000,2.855000,5.900000,15.600000,8.600000,15.500000,20.800000
23,80.900000,1.936667,2.318000,3.900000,12.300000,7.100000,15.100000,19.100000
